# Deliverable 4 - INF201 

Submission deadline: 27th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

## Task 1: Temperatures

Create a `Temperature` class that supports Celsius ("C"), Fahrenheit ("F"), and Kelvin ("K"). This class should:
1. Be initialised with a value and a unit (case-insensitive), and raise a `ValueError` if the unit is not one of "C", "F" or "K", or if the temperature is below absolute zero (0 K).
2. Use a tolerance of 1e-9 for floating-point value comparisons
3. Expose `value` and `unit` as read-only attributes
4. Contain the following conversion methods:
    * `to_celsius(self) -> float`:
        * From Kelvin: $C = K - 273.15$
        * From Fahrenheit: $C = (F - 32) \times \frac{5}{9}$
    * `to_fahrenheit(self) -> float`:
        * From Celsius: $F = C \times \frac{9}{5} + 32$
        * From Kelvin: $F = (K - 273.15) \times \frac{9}{5} + 32$
    * `to_kelvin(self) -> float`:
        * From Celsius: $K = C + 273.15$
        * From Fahrenheit: $K = (F - 32) \times \frac{5}{9} + 273.15$
    * `convert_to(self, target_unit: str) -> Temperature`:
        * Returns a new `Temperature` instance converted to `target_unit` (case-insensitive), raising `ValueError` if `target_unit` is invalid.
5. Contain the following string representations:
    * `__repr__(self) -> str`: Returns an unambiguous representation, such as `Temperature(25.0, 'C')`.
    * `__str__(self) -> str`: Returns a human-readable string formatted to two decimal places, such as `25.00 °C`.
6. Contain all six comparisons (==, !=, <, <=, >, >=) so temperatures can be compared across different units (`Temperature(0, "C") == Temperature(32, "F")`). Should return `NotImplemented` if a `Temperature` instance is compared to a non-Temperature instance.

Write a `test_temperature.py` script that imports the `Temperature` class and uses pytest for the following tests:
1. Initialization: Default unit, case-insensitive units, and read-only behavior.
2. Validation: Invalid unit strings and temperatures below absolute zero in "C", "F", and "K".
3. Conversions: Test conversion accuracy across reference points:
    * Absolute zero ($-273.15^\circ\text{C} = -459.67^\circ\text{F} = 0\text{ K}$)
    * Celsius–Fahrenheit crossover ($-40^\circ\text{C} = -40^\circ\text{F} = 233.15\text{ K}$)
    * Freezing point of water ($0^\circ\text{C} = 32^\circ\text{F} = 273.15\text{ K}$)
    * Human body temperature ($37^\circ\text{C} = 98.6^\circ\text{F} = 310.15\text{ K}$)
    * Boiling point of water ($100^\circ\text{C} = 212^\circ\text{F} = 373.15\text{ K}$)
4. Round-trip conversions: Converting "C" -> "F" -> "K" -> "C" reproduces the original Temperature.
5. Cross-unit comparisons: Equality and ordering (<, <=, >, >=) across mixed units.
6. Formatting: Outputs of `str()` and `repr()`

### Our solution

The `Temperature` class is implemented in `temperature.py`, and the tests are in `test_temperature.py`.

* The unit is case-insensitive and stored in upper case. A `ValueError` is raised if the unit is not "C", "F" or "K", or if the value is below absolute zero.
* The absolute-zero check converts the value to Kelvin first, so the same rule works for all three units. A tolerance of 1e-9 is used so that values like $-273.15^\circ\text{C}$ (exactly absolute zero) are not rejected because of floating-point rounding.
* `value` and `unit` are stored in the protected attributes `_value` and `_unit` and exposed through `@property` without a setter, which makes them read-only.
* `convert_to` validates the target unit, picks the matching conversion method and returns a new `Temperature` object.
* All six comparisons convert both temperatures to Kelvin, so different units can be compared directly. `==` and `!=` treat values within 1e-9 of each other as equal, and the ordering operators use the same tolerance, so that for example `Temperature(0, "C") <= Temperature(32, "F")` is `True`.
* Comparing with a non-`Temperature` returns `NotImplemented`, which lets Python raise a `TypeError` for `<`, `<=`, `>` and `>=`, and return `False` for `==`.

In [1]:
from temperature import Temperature

t = Temperature(25, "c")
print(repr(t))
print(t)
print(f"{t} = {t.to_fahrenheit():.2f} °F = {t.to_kelvin():.2f} K")
print(t.convert_to("K"))

print(f"0 °C == 32 °F: {Temperature(0, 'C') == Temperature(32, 'F')}")
print(f"-40 °C == -40 °F: {Temperature(-40, 'C') == Temperature(-40, 'F')}")
print(f"300 K > 25 °C: {Temperature(300, 'K') > Temperature(25, 'C')}")

try:
    Temperature(-300, "C")
except ValueError as e:
    print(f"ValueError: {e}")

Temperature(25.0, 'C')
25.00 °C
25.00 °C = 77.00 °F = 298.15 K
298.15 K
0 °C == 32 °F: True
-40 °C == -40 °F: True
300 K > 25 °C: True
ValueError: -300.0 C is below absolute zero (0 K).


In [2]:
import sys
!{sys.executable} -m pytest test_temperature.py -v --color=no

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-9.2.0, pluggy-1.7.0 -- /Users/eiriknysveen/Documents/INF201-deliverable4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/eiriknysveen/Documents/INF201-deliverable4
plugins: platformdirs-4.14.8, anyio-4.16.0
collected 85 items                                                             

test_temperature.py::test_default_unit_is_celsius PASSED                 [  1%]
test_temperature.py::test_unit_is_case_insensitive[c-C] PASSED           [  2%]
test_temperature.py::test_unit_is_case_insensitive[C-C] PASSED           [  3%]
test_temperature.py::test_unit_is_case_insensitive[f-F] PASSED           [  4%]
test_temperature.py::test_unit_is_case_insensitive[F-F] PASSED           [  5%]
test_temperature.py::test_unit_is_case_insensitive[k-K] PASSED           [  7%]
test_temperature.py::test_unit_is_case_insensitive[K-K] PASSED           [  8%]
test_temperature.py::

## Task 2: Fractions

Create a `Fraction` class representing fractions. Both numerator and denominator should be integers, and the denominator should be stricktly positive (>0). The class should implement the following special methods (aka magic methods, dunder methods): 
* initialiser
* all comparisons (equal, not equal, smaller than, smaller than or equal, greater than, and greated than or equal)
* at least four other special methods

Write a test script to be used with `pytest` for testing the implemented methods. Include the output from testing in a Jupyter notebok.

### Our solution

The `Fraction` class is implemented in `fraction.py`, and the tests are in `test_fraction.py`.

* The fraction is always stored in reduced form using `math.gcd`, so `Fraction(2, 4)` becomes `1/2`.
* A `TypeError` is raised if the numerator or denominator is not an integer, and a `ValueError` if the denominator is not strictly positive.
* Comparisons use cross-multiplication: $\frac{a}{b} < \frac{c}{d} \iff a \cdot d < c \cdot b$ (this works because both denominators are positive).
* Comparing with a non-`Fraction` returns `NotImplemented`.

Special methods implemented: `__init__`, `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__str__` and `__repr__`.

In [3]:
from fraction import Fraction

a = Fraction(1, 2)
b = Fraction(3, 4)

print(f"{a} + {b} = {a + b}")
print(f"{a} - {b} = {a - b}")
print(f"{a} * {b} = {a * b}")
print(f"{a} / {b} = {a / b}")
print(f"{a} < {b}: {a < b}")
print(f"Fraction(2, 4) == Fraction(1, 2): {Fraction(2, 4) == Fraction(1, 2)}")
print(repr(Fraction(6, 8)))

1/2 + 3/4 = 5/4
1/2 - 3/4 = -1/4
1/2 * 3/4 = 3/8
1/2 / 3/4 = 2/3
1/2 < 3/4: True
Fraction(2, 4) == Fraction(1, 2): True
Fraction(3, 4)


In [4]:
import sys
!{sys.executable} -m pytest test_fraction.py -v --color=no

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-9.2.0, pluggy-1.7.0 -- /Users/eiriknysveen/Documents/INF201-deliverable4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/eiriknysveen/Documents/INF201-deliverable4
plugins: platformdirs-4.14.8, anyio-4.16.0
collected 24 items                                                             

test_fraction.py::test_init PASSED                                       [  4%]
test_fraction.py::test_default_denominator PASSED                        [  8%]
test_fraction.py::test_fraction_is_reduced PASSED                        [ 12%]
test_fraction.py::test_denominator_must_be_positive[0] PASSED            [ 16%]
test_fraction.py::test_denominator_must_be_positive[-3] PASSED           [ 20%]
test_fraction.py::test_must_be_integers[1.5-2] PASSED                    [ 25%]
test_fraction.py::test_must_be_integers[1-2.0] PASSED                    [ 29%]
test_fraction.py::tes